# DWM Lab 7 — K-Means Customer Clustering

### Dataset: Online Retail

**Objective:**  
To perform **customer segmentation using the K-Means clustering algorithm** on the Online Retail dataset.

The experiment includes:
- Customer-level feature creation
- Feature scaling
- Elbow Method
- K-Means clustering
- Cluster visualization
- Cluster-wise analysis

### Dataset adaptation

The original lab performs customer clustering using online retail transaction data.  
Here, the same **Online Retail.xlsx** dataset is used to create customer-level behavioural features.

The main features used for clustering are:

- **TotalSpending** — total amount spent by the customer
- **TotalOrders** — number of unique invoices/orders
- **TotalQuantity** — total quantity purchased
- **AverageOrderValue** — average spending per order
- **UniqueProducts** — number of different products purchased

## 1. Import Required Libraries

Pandas and NumPy are used for data preparation. Matplotlib and Seaborn are used for visualization. Scikit-learn provides StandardScaler and KMeans.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")

## 2. Load the Dataset

Load the same `Online Retail.xlsx` dataset used in the previous DWM experiments.

In [ ]:
file_path = "Online Retail.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Data Preprocessing

For customer segmentation:

1. Convert `InvoiceDate` to datetime.
2. Remove cancelled invoices.
3. Remove transactions with non-positive quantity or price.
4. Calculate `SalesAmount`.
5. Remove records without `CustomerID`, because customer-level clustering requires a customer identifier.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Keep valid transactions
df = df[
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0)
].copy()

# Calculate transaction revenue
df['SalesAmount'] = df['Quantity'] * df['UnitPrice']

# Customer ID is required for customer segmentation
df = df.dropna(subset=['CustomerID']).copy()

# Use a consistent customer ID representation
df['CustomerID'] = df['CustomerID'].astype(int)

print("Cleaned transaction shape:", df.shape)
display(df.head())

## 4. Create Customer-Level Features

K-Means requires one row per observation.  
Since the original data contains transaction-level records, transactions are aggregated to the customer level.

For every customer we calculate:
- Total spending
- Number of orders
- Total quantity purchased
- Average order value
- Number of unique products purchased

In [ ]:
customer_df = df.groupby('CustomerID').agg(
    TotalSpending=('SalesAmount', 'sum'),
    TotalOrders=('InvoiceNo', 'nunique'),
    TotalQuantity=('Quantity', 'sum'),
    AverageOrderValue=('SalesAmount', lambda x: x.sum() / df.loc[x.index, 'InvoiceNo'].nunique()),
    UniqueProducts=('StockCode', 'nunique')
).reset_index()

print("Number of customers:", len(customer_df))
display(customer_df.head())

## 5. Examine Customer Features

Use descriptive statistics to understand the customer-level data before clustering.

In [ ]:
customer_df.describe().T

## 6. Handle Skewed Values

Retail spending and quantity can contain very large values.  
A log transformation is applied to the positive customer features to reduce the influence of extreme values before scaling.

`log1p(x)` calculates `log(1+x)` and safely handles zero values.

In [ ]:
features = [
    'TotalSpending',
    'TotalOrders',
    'TotalQuantity',
    'AverageOrderValue',
    'UniqueProducts'
]

customer_features = customer_df[features].copy()

# Log transform
customer_features_log = np.log1p(customer_features)

display(customer_features_log.head())

## 7. Feature Scaling

K-Means uses distances to assign observations to clusters.  
Therefore, features with larger numerical scales should not dominate the clustering.

StandardScaler transforms each feature approximately to:

**mean = 0 and standard deviation = 1**

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(customer_features_log)

print("Scaled feature matrix shape:", X_scaled.shape)

## 8. Elbow Method

The **Within-Cluster Sum of Squares (WCSS)**, also called inertia, is calculated for different values of K.

The elbow point gives an indication of a suitable number of clusters.

In [ ]:
wcss = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_values, wcss, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS / Inertia')
plt.xticks(k_values)
plt.tight_layout()
plt.show()

elbow_table = pd.DataFrame({
    'K': list(k_values),
    'WCSS': wcss
})

display(elbow_table)

## 9. Select K and Apply K-Means

For this experiment, **K = 3** is selected to create three customer segments.

The K-Means algorithm:
1. Initializes cluster centroids.
2. Assigns each customer to the nearest centroid.
3. Recalculates centroids.
4. Repeats until the assignments stabilize.

In [ ]:
optimal_k = 3

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

customer_df['Cluster'] = kmeans.fit_predict(X_scaled)

print("K-Means clustering completed.")
print("\nCluster counts:")
print(customer_df['Cluster'].value_counts().sort_index())

## 10. Evaluate Clustering Using Silhouette Score

The Silhouette Score measures how similar an observation is to its own cluster compared with other clusters.

Its value generally ranges from **-1 to +1**.

A higher value indicates better separation between clusters.

In [ ]:
silhouette = silhouette_score(X_scaled, customer_df['Cluster'])

print("Silhouette Score:", silhouette)

## 11. Analyze the Customer Clusters

Calculate the average customer characteristics for each cluster.

In [ ]:
cluster_summary = customer_df.groupby('Cluster')[features].mean().round(2)

print("Average characteristics of each cluster:")
display(cluster_summary)

## 12. Cluster Size

Visualize how many customers belong to each cluster.

In [ ]:
cluster_counts = customer_df['Cluster'].value_counts().sort_index()

plt.figure(figsize=(8, 5))
cluster_counts.plot(kind='bar')
plt.title('Number of Customers in Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 13. Customer Segmentation — Spending vs Orders

Visualize the clusters using total spending and total orders.

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=customer_df,
    x='TotalOrders',
    y='TotalSpending',
    hue='Cluster',
    palette='deep',
    alpha=0.65
)

plt.title('Customer Segmentation: Total Orders vs Total Spending')
plt.xlabel('Total Orders')
plt.ylabel('Total Spending')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## 14. Customer Segmentation — Quantity vs Spending

A second visualization compares total quantity purchased with total spending.

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=customer_df,
    x='TotalQuantity',
    y='TotalSpending',
    hue='Cluster',
    palette='deep',
    alpha=0.65
)

plt.title('Customer Segmentation: Quantity vs Spending')
plt.xlabel('Total Quantity')
plt.ylabel('Total Spending')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## 15. Cluster-wise Customer Analysis

The table below provides a detailed summary of each segment.

This can be used to describe the clusters as groups such as:
- lower-spending customers
- regular customers
- high-value customers

The actual interpretation should be based on the calculated cluster averages.

In [ ]:
cluster_analysis = customer_df.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    AvgSpending=('TotalSpending', 'mean'),
    AvgOrders=('TotalOrders', 'mean'),
    AvgQuantity=('TotalQuantity', 'mean'),
    AvgOrderValue=('AverageOrderValue', 'mean'),
    AvgUniqueProducts=('UniqueProducts', 'mean')
).round(2)

display(cluster_analysis)

## Conclusion

Customer segmentation was performed using the **K-Means clustering algorithm** on the Online Retail dataset.

The experiment included:

- Data cleaning
- Customer-level feature engineering
- Total spending, order count, quantity, average order value and unique-product features
- Log transformation
- Standardization
- Elbow Method
- K-Means clustering with K = 3
- Silhouette Score
- Cluster size analysis
- Cluster visualization
- Cluster-wise customer analysis

The resulting customer-level dataset can also be reused for the next clustering experiment using **Agglomerative Hierarchical Clustering**.